# Myllia: Bilinear v3 (GenePT → Expression-SVD Alignment + Metric Loss + Baseline Blending)

This notebook learns an **alignment** from **GenePT embeddings → expression-derived SVD gene embeddings** built from `training_cells.h5ad`.

Why: raw GenePT space often does not match the expression geometry of this dataset, so swapping it in directly can degrade generalization.

What you get:
1) Metric-aware loss: `weighted_l1_like + λ * weighted_cosine_loss`
2) Learned amplitude blending: `y = s(g)*pred + (1-s(g))*delta_baseline`
3) GenePT alignment (recommended): learn `W` such that `W(genept(g)) ≈ svd(g)` using the 5127 output genes.

If GenePT download/load fails, everything falls back to pure SVD embeddings.


In [1]:
# -----------------------------
# Imports & global settings
# -----------------------------
import os
import zipfile
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

import anndata as ad
import scanpy as sc
from scipy import sparse

from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

import torch
import torch.nn as nn

from myllia_metric import myllia_score

SEED = 6
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(DEVICE)

ROOT = Path(".")

# -----------------------------
# Main knobs
# -----------------------------
RANK_R       = 32
EMB_DIM_OUT  = 128   # output-gene embedding dim (SVD space)
EMB_DIM_PERT = 128   # pert-gene embedding dim fed into model (aligned space)

# Loss mix
LAMBDA_COS   = 0.35
GATE_A       = 0.0
GATE_B       = 0.2
EPS          = 1e-12

# Training
DROPOUT      = 0.10
LR           = 2e-3
WD           = 1e-4
EPOCHS       = 1200
BATCH_GENES  = 16
EVAL_EVERY   = 25
PATIENCE     = 14

# Embedding policy
USE_GENEPT_PERT = True      # use aligned GenePT for perturbation genes when available
ALIGN_GENEPT_TO_SVD = True  # learn mapping GenePT -> SVD (recommended)
GENEPT_PCA_DIM = None       # optionally reduce GenePT dim before alignment (e.g. 512). None = no PCA.

# GenePT download (best-effort)
GENEPT_URL = "https://zenodo.org/records/10833191/files/GenePT_emebdding_v2.zip?download=1"
GENEPT_DIR = ROOT / "external" / "genept"
GENEPT_DIR.mkdir(parents=True, exist_ok=True)

print("device:", device)


device: cuda


In [2]:
# -----------------------------
# Metric scoring helper
# -----------------------------
def score_delta(dt: np.ndarray, dp: np.ndarray) -> dict:
    dt = dt.astype(np.float32, copy=False)
    dp = dp.astype(np.float32, copy=False)
    r = myllia_score(dt, dp)
    return {
        "score": float(r.score),
        "wcos": float(r.wcos),
        "mean_term": float(r.mean_term),
        "pred_wmae": float(r.pred_wmae),
    }


In [3]:
# -----------------------------
# Load competition data
# -----------------------------
means_path = ROOT / "data" / "training_data_means.csv"
valmap_path = ROOT / "data" / "pert_ids_val.csv"
sample_sub_path = ROOT / "data" / "sample_submission.csv"

df_means = pd.read_csv(means_path)
df_valmap = pd.read_csv(valmap_path)
df_sub = pd.read_csv(sample_sub_path)

gene_columns = [c for c in df_means.columns if c != "pert_symbol"]

baseline_mask = df_means["pert_symbol"].astype(str) == "non-targeting"
x_base = df_means.loc[baseline_mask, gene_columns].iloc[0].to_numpy(np.float32)

df_train = df_means.loc[~baseline_mask].reset_index(drop=True)
train_genes = df_train["pert_symbol"].astype(str).to_numpy()

X_train_means = df_train[gene_columns].to_numpy(np.float32)
D_train = (X_train_means - x_base[None, :]).astype(np.float32)  # (80, 5127)

delta_baseline = D_train.mean(axis=0).astype(np.float32)

val_map = dict(zip(df_valmap["pert_id"].astype(str), df_valmap["pert"].astype(str)))

print("Train perts:", len(train_genes), "G:", len(gene_columns))
print("Sample submission rows:", len(df_sub))
print("Val mapping entries:", len(val_map))


Train perts: 80 G: 5127
Sample submission rows: 120
Val mapping entries: 60


In [4]:
# -----------------------------
# Build SVD embeddings from training_cells.h5ad (expression-native space)
# Output embeddings ALWAYS come from here.
# -----------------------------
def find_h5ad():
    candidates = [
        ROOT / "data" / "training_cells.h5ad",
        ROOT / "Data" / "training_cells.h5ad",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("training_cells.h5ad not found in data/ or Data/")

def build_svd_embeddings_from_h5ad(h5ad_path: Path, genes: list[str], k: int, seed: int):
    adata = ad.read_h5ad(str(h5ad_path)).copy()

    # normalize on ALL genes first (matches competition normalization intent)
    sc.pp.normalize_total(adata, target_sum=1e4, inplace=True)

    varU = pd.Index([str(v).upper() for v in adata.var_names])
    genesU = [str(g).upper() for g in genes]
    pos = varU.get_indexer(genesU)
    ok = pos >= 0

    genes_ok = [genesU[i] for i in range(len(genesU)) if ok[i]]
    pos_ok = pos[ok]

    missing = [genesU[i] for i in range(len(genesU)) if not ok[i]]
    if missing:
        print(f"[warn] {len(missing)} / {len(genesU)} genes missing from h5ad. Example: {missing[:12]}")

    adata = adata[:, pos_ok].copy()

    X = adata.X
    if not sparse.issparse(X):
        X = sparse.csr_matrix(X)
    else:
        X = X.tocsr(copy=True)

    X.data = np.log2(X.data + 1.0).astype(np.float32)

    svd = TruncatedSVD(n_components=k, random_state=seed)
    svd.fit(X)

    gene_emb = svd.components_.T.astype(np.float32)  # (n_genes_ok, k)
    gene2emb = {genes_ok[i]: gene_emb[i] for i in range(len(genes_ok))}
    fallback = gene_emb.mean(axis=0).astype(np.float32)
    return gene2emb, fallback

h5ad_path = find_h5ad()

val_targets = df_valmap["pert"].astype(str).tolist()
union_genes = sorted(set([g.upper() for g in gene_columns] +
                         [g.upper() for g in train_genes.tolist()] +
                         [g.upper() for g in val_targets]))

print("union_genes:", len(union_genes))

gene2svd, svd_fallback = build_svd_embeddings_from_h5ad(
    h5ad_path=h5ad_path,
    genes=union_genes,
    k=max(EMB_DIM_OUT, EMB_DIM_PERT),
    seed=SEED
)

def svd_emb(g: str, d: int) -> np.ndarray:
    v = gene2svd.get(str(g).upper(), svd_fallback)
    return v[:d].copy()

U_out = np.vstack([svd_emb(g, EMB_DIM_OUT) for g in gene_columns]).astype(np.float32)
print("U_out:", U_out.shape)


union_genes: 5143
U_out: (5127, 128)


In [5]:
# -----------------------------
# GenePT loader (best-effort)
# Expects a pickle containing dict gene->vector.
# -----------------------------
def _download(url: str, dst: Path, force: bool = False):
    import urllib.request
    if dst.exists() and not force:
        return
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "Accept": "*/*",
        "Connection": "keep-alive",
    }
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req) as r, open(dst, "wb") as f:
        f.write(r.read())

def load_genept_dict():
    zip_path = GENEPT_DIR / "GenePT_embedding_v2.zip"
    extract_dir = GENEPT_DIR / "extracted"
    extract_dir.mkdir(parents=True, exist_ok=True)

    try:
        if not zip_path.exists():
            print("[genept] downloading...")
            _download(GENEPT_URL, zip_path, force=False)
        print("[genept] zip:", zip_path)
    except Exception as e:
        print("[genept] download failed:", repr(e))
        return None

    try:
        marker = extract_dir / ".done"
        if not marker.exists():
            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall(extract_dir)
            marker.write_text("ok", encoding="utf-8")
    except Exception as e:
        print("[genept] unzip failed:", repr(e))
        return None

    pkl_files = []
    for root, _, files in os.walk(extract_dir):
        for fn in files:
            lo = fn.lower()
            if lo.endswith(".pickle") or lo.endswith(".pkl"):
                pkl_files.append(Path(root) / fn)

    if not pkl_files:
        print("[genept] no pickle found")
        return None

    chosen = None
    for p in pkl_files:
        name = p.name.lower()
        if "model_3" in name or "model-3" in name or "m3" in name:
            chosen = p
            break
    if chosen is None:
        chosen = pkl_files[0]

    print("[genept] using pickle:", chosen)

    try:
        with open(chosen, "rb") as f:
            obj = pickle.load(f)
    except Exception as e:
        print("[genept] pickle load failed:", repr(e))
        return None

    if not isinstance(obj, dict):
        print("[genept] expected dict gene->vec, got:", type(obj))
        return None

    out = {}
    for k, v in obj.items():
        if k is None:
            continue
        kk = str(k).upper()
        vv = np.asarray(v, dtype=np.float32).ravel()
        out[kk] = vv
    return out

genept_dict = load_genept_dict()
if genept_dict is None:
    print("[genept] unavailable -> using pure SVD")
else:
    some = next(iter(genept_dict.values()))
    print("[genept] dict size:", len(genept_dict), "dim example:", some.shape)


[genept] zip: external\genept\GenePT_embedding_v2.zip
[genept] using pickle: external\genept\extracted\GenePT_emebdding_v2\GenePT_gene_protein_embedding_model_3_text.pickle
[genept] dict size: 122750 dim example: (3072,)


In [6]:
# -----------------------------
# Build GenePT -> SVD alignment
# Fit ridge: (standardized GenePT) -> (SVD 128)
# -----------------------------
genept_scaler = None
genept_pca = None
genept_to_svd = None
genept_fallback_raw = None
aligned_fallback = None

def build_alignment():
    global genept_scaler, genept_pca, genept_to_svd, genept_fallback_raw, aligned_fallback
    if genept_dict is None:
        return False

    out_genesU = [g.upper() for g in gene_columns]
    X_list = []
    Y_list = []

    for g in out_genesU:
        v = genept_dict.get(g, None)
        if v is None:
            continue
        X_list.append(v)
        Y_list.append(svd_emb(g, EMB_DIM_PERT))

    if len(X_list) < 2000:
        print(f"[align] overlap too small: {len(X_list)}")
        return False

    X = np.vstack(X_list).astype(np.float32)
    Y = np.vstack(Y_list).astype(np.float32)

    print("[align] overlap:", X.shape, Y.shape)

    genept_fallback_raw = X.mean(axis=0).astype(np.float32)

    genept_scaler = StandardScaler(with_mean=True, with_std=True)
    Xs = genept_scaler.fit_transform(X)

    if GENEPT_PCA_DIM is not None and GENEPT_PCA_DIM < Xs.shape[1]:
        genept_pca = PCA(n_components=GENEPT_PCA_DIM, random_state=SEED)
        Xz = genept_pca.fit_transform(Xs).astype(np.float32)
        print("[align] genept PCA:", Xs.shape[1], "->", Xz.shape[1])
    else:
        genept_pca = None
        Xz = Xs.astype(np.float32)

    genept_to_svd = Ridge(alpha=10.0, fit_intercept=True, random_state=SEED)
    genept_to_svd.fit(Xz, Y)

    aligned_fallback = genept_to_svd.predict(Xz.mean(axis=0, keepdims=True)).astype(np.float32)[0]

    r2 = genept_to_svd.score(Xz, Y)
    print("[align] ridge R^2 (train overlap):", float(r2))
    return True

ALIGN_OK = False
if ALIGN_GENEPT_TO_SVD:
    ALIGN_OK = build_alignment()
    if not ALIGN_OK:
        print("[align] failed -> disabling GenePT pert usage")
        USE_GENEPT_PERT = False
else:
    USE_GENEPT_PERT = False
    print("[align] disabled")


[align] overlap: (4998, 3072) (4998, 128)
[align] ridge R^2 (train overlap): 0.6136360168457031


In [7]:
# -----------------------------
# Pert embedding function (aligned GenePT if enabled, else SVD)
# -----------------------------
def genept_raw(g: str) -> np.ndarray:
    if genept_dict is None:
        return None
    v = genept_dict.get(str(g).upper(), None)
    if v is None:
        return genept_fallback_raw
    return v

def aligned_genept(g: str) -> np.ndarray:
    v = genept_raw(g)
    if v is None:
        return aligned_fallback
    v = v.astype(np.float32, copy=False)[None, :]
    xs = genept_scaler.transform(v)
    if genept_pca is not None:
        xs = genept_pca.transform(xs)
    y = genept_to_svd.predict(xs).astype(np.float32)[0]
    return y

def pert_emb(g: str) -> np.ndarray:
    gu = str(g).upper()
    if USE_GENEPT_PERT and ALIGN_OK:
        try:
            return aligned_genept(gu)[:EMB_DIM_PERT].copy()
        except Exception:
            return svd_emb(gu, EMB_DIM_PERT)
    return svd_emb(gu, EMB_DIM_PERT)

Z_train = np.vstack([pert_emb(g) for g in train_genes]).astype(np.float32)
print("Z_train:", Z_train.shape, "USE_GENEPT_PERT:", USE_GENEPT_PERT, "ALIGN_OK:", ALIGN_OK)


Z_train: (80, 128) USE_GENEPT_PERT: True ALIGN_OK: True


In [8]:
# -----------------------------
# Loss: weighted L1-like + weighted cosine
# -----------------------------
def gate_smoothstep(x: torch.Tensor, a: float = GATE_A, b: float = GATE_B) -> torch.Tensor:
    t = (x - a) / (b - a)
    t = torch.clamp(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)

def weighted_l1_like(dt: torch.Tensor, dp: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    w = gate_smoothstep(torch.abs(dt), a=GATE_A, b=GATE_B)
    err = torch.abs(dp - dt)
    num = torch.sum(w * err, dim=1)
    den = torch.clamp(torch.sum(w, dim=1), min=eps)
    return torch.mean(num / den)

def weighted_cosine_loss(dt: torch.Tensor, dp: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    w = gate_smoothstep(torch.abs(dt), a=GATE_A, b=GATE_B)
    num = torch.sum(w * dt * dp, dim=1)
    den = torch.clamp(torch.sqrt(torch.sum(w * dt * dt, dim=1)) * torch.sqrt(torch.sum(w * dp * dp, dim=1)), min=eps)
    cos = num / den
    return torch.mean(1.0 - cos)

def total_loss(dt: torch.Tensor, dp: torch.Tensor) -> torch.Tensor:
    return weighted_l1_like(dt, dp, eps=EPS) + (LAMBDA_COS * weighted_cosine_loss(dt, dp, eps=EPS))


In [9]:
# -----------------------------
# Model: bilinear + learned blending with baseline
# -----------------------------
class BilinearDeltaModelV3(nn.Module):
    def __init__(self, d_pert: int, d_out: int, rank_r: int, dropout: float):
        super().__init__()
        self.proj_p = nn.Sequential(
            nn.Linear(d_pert, rank_r),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.proj_o = nn.Sequential(
            nn.Linear(d_out, rank_r),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.bias_global = nn.Parameter(torch.zeros(1))
        self.bias_gene = None

        self.scale = nn.Sequential(
            nn.Linear(d_pert, 64),
            nn.GELU(),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def set_gene_bias(self, G):
        dev = next(self.parameters()).device
        self.bias_gene = torch.nn.Parameter(torch.zeros(G, device=dev))
        self.bias_global = torch.nn.Parameter(torch.zeros(1, device=dev))

    def forward(self, z_pert: torch.Tensor, u_out: torch.Tensor, baseline: torch.Tensor) -> torch.Tensor:
        p = self.proj_p(z_pert)   # (B, R)
        o = self.proj_o(u_out)    # (G, R)
        y = p @ o.T               # (B, G)
        y = y + self.bias_gene[None, :] + self.bias_global

        s = self.scale(z_pert)    # (B, 1)
        y = (s * y) + ((1.0 - s) * baseline[None, :])
        return y


In [10]:
# -----------------------------
# Prepare tensors
# -----------------------------
Y = D_train.astype(np.float32)
G = Y.shape[1]
N = Y.shape[0]

Uo_t = torch.tensor(U_out, device=device)
Zt = torch.tensor(Z_train, device=device)
Yt = torch.tensor(Y, device=device)
baseline_t = torch.tensor(delta_baseline.astype(np.float32), device=device)

print("N:", N, "G:", G, "device:", device)


N: 80 G: 5127 device: cuda


In [11]:
# -----------------------------
# CV training (gene-level KFold), early stop on official metric
# -----------------------------
def train_one_fold(tr_idx, va_idx):
    model = BilinearDeltaModelV3(EMB_DIM_PERT, EMB_DIM_OUT, RANK_R, DROPOUT).to(device)
    model.set_gene_bias(G)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    tr_idx = np.asarray(tr_idx)
    va_idx = np.asarray(va_idx)
    va_idx_t = torch.tensor(va_idx, device=device, dtype=torch.long)

    best_score = -1e18
    best_state = None
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = tr_idx.copy()
        np.random.shuffle(perm)

        for start in range(0, len(perm), BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo_t, baseline_t)
            loss = total_loss(Yt.index_select(0, b_t), pred)

            opt.zero_grad()
            loss.backward()
            opt.step()

        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                va_pred = model(Zt.index_select(0, va_idx_t), Uo_t, baseline_t).detach().cpu().numpy().astype(np.float32)
            va_true = Y[va_idx]

            s = score_delta(va_true, va_pred)
            sc = s["score"]

            if sc > best_score:
                best_score = sc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    return best_score, best_state

kf = KFold(n_splits=8, shuffle=True, random_state=SEED)
fold_scores = []
for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(N)), 1):
    best_score, _ = train_one_fold(tr_idx, va_idx)
    fold_scores.append(float(best_score))
    print(f"fold {fold}: best_score={best_score:.6f}")

print(f"[cv] mean={float(np.mean(fold_scores)):.6f} std={float(np.std(fold_scores)):.6f}")


fold 1: best_score=0.069951
fold 2: best_score=0.057333
fold 3: best_score=0.048339
fold 4: best_score=0.067551
fold 5: best_score=0.101568
fold 6: best_score=0.086798
fold 7: best_score=0.090887
fold 8: best_score=0.050019
[cv] mean=0.071556 std=0.018475


In [12]:
# -----------------------------
# Fit final model on all 80, write submission
# -----------------------------
def fit_full_model():
    model = BilinearDeltaModelV3(EMB_DIM_PERT, EMB_DIM_OUT, RANK_R, DROPOUT).to(device)
    model.set_gene_bias(G)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    best_score = -1e18
    best_state = None
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = np.arange(N)
        np.random.shuffle(perm)

        for start in range(0, N, BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo_t, baseline_t)
            loss = total_loss(Yt.index_select(0, b_t), pred)

            opt.zero_grad()
            loss.backward()
            opt.step()

        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                pred_np = model(Zt, Uo_t, baseline_t).detach().cpu().numpy().astype(np.float32)
            s = score_delta(Y, pred_np)
            sc = s["score"]
            print(f"epoch={epoch:4d} train_score={sc:.6f} wcos={s['wcos']:.6f} pred_wmae={s['pred_wmae']:.6f}")

            if sc > best_score:
                best_score = sc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    return model

model_final = fit_full_model()

def predict_delta_gene(gene_symbol: str) -> np.ndarray:
    z = torch.tensor(pert_emb(gene_symbol)[None, :].astype(np.float32), device=device)
    with torch.no_grad():
        y = model_final(z, Uo_t, baseline_t).detach().cpu().numpy().astype(np.float32)[0]
    return y

sub = df_sub.copy()
sub["pert_id"] = sub["pert_id"].astype(str)
sub_gene_cols = [c for c in sub.columns if c != "pert_id"]

idx = {g: i for i, g in enumerate(gene_columns)}
perm = [idx[g] for g in sub_gene_cols]

sub.loc[:, sub_gene_cols] = np.tile(delta_baseline[perm][None, :], (len(sub), 1))

hit = 0
for pid, gene in val_map.items():
    vec = predict_delta_gene(gene)[perm]
    m = (sub["pert_id"] == str(pid))
    if m.any():
        sub.loc[m, sub_gene_cols] = vec[None, :]
        hit += int(m.sum())

print(f"[ok] filled {hit} leaderboard rows from mapping")

out_csv = ROOT / "submission_bilinear_v3_genept_align.csv"
sub.to_csv(out_csv, index=False)
print("[ok] wrote:", out_csv)


epoch=  25 train_score=0.095829 wcos=0.560532 pred_wmae=0.081590
epoch=  50 train_score=0.136684 wcos=0.610528 pred_wmae=0.079478
epoch=  75 train_score=0.139102 wcos=0.680776 pred_wmae=0.080012
epoch= 100 train_score=0.090152 wcos=0.767906 pred_wmae=0.083045
epoch= 125 train_score=0.087274 wcos=0.807379 pred_wmae=0.083308
epoch= 150 train_score=0.095222 wcos=0.829997 pred_wmae=0.082993
epoch= 175 train_score=0.107988 wcos=0.844556 pred_wmae=0.082430
epoch= 200 train_score=0.123933 wcos=0.855074 pred_wmae=0.081732
epoch= 225 train_score=0.146941 wcos=0.862075 pred_wmae=0.080702
epoch= 250 train_score=0.171513 wcos=0.866864 pred_wmae=0.079626
epoch= 275 train_score=0.203188 wcos=0.869261 pred_wmae=0.078231
epoch= 300 train_score=0.241452 wcos=0.870814 pred_wmae=0.076518
epoch= 325 train_score=0.286066 wcos=0.871462 pred_wmae=0.074430
epoch= 350 train_score=0.333206 wcos=0.871610 pred_wmae=0.072110
epoch= 375 train_score=0.387748 wcos=0.871609 pred_wmae=0.069420
epoch= 400 train_score=0.